In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

from src.utils import *

import time
import git_root
import importlib
import subprocess
from tqdm import tqdm
import ipyparallel as ipp

## Setup Engines

In [ ]:
profile = "default"

subprocess.call("pkill -u $USER -f ipcontroller", shell=True)
subprocess.call("pkill -u $USER -f ipengine", shell=True)
time.sleep(3)

cmd = f"ipcontroller --profile={profile} --ip='*' --log-to-file --log-level=20 &"
print("Starting controller:")
print("Command:", cmd)
subprocess.call(cmd, shell=True)

time.sleep(5)

rc = ipp.Client(profile=profile)
print("Controller connected.")
print("Current engine IDs:", rc.ids)

In [ ]:
cmd_slist_cores = """slist | awk '$1 == "physics" {print $6}'"""
slist_cores_result = subprocess.run(cmd_slist_cores, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, universal_newlines=True)
CORES_AVALIBLE = int(slist_cores_result.stdout)

print("Cores avalible:", CORES_AVALIBLE)

In [ ]:
N_ENGINES = 24 # add max the numbers of cores avalible or else will sit in queue. 

# WALLTIME = "336:00:00" # 14 Day - Maximum Limit on Purdue Negishi & Bell Clusters
WALLTIME = "168:00:00" # 7 Days

repo = git_root.git_root()
cmd = f"sbatch --ntasks={N_ENGINES} --time={WALLTIME} {git_root.git_root()}/scripts/start_ipengines.sh"

result = subprocess.run(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, universal_newlines=True)

print("Command:", cmd)
print("STDOUT:", result.stdout)
if (result.stderr != ""):
    print("STDERR:", result.stderr)
print("Return code:", result.returncode)

In [ ]:
rc = ipp.Client(profile="default") # creates client connected to IPyParallel controller with "default" profile
print("Initial IDs:", rc.ids) # which engines are already connected at that moment

rc.wait_for_engines(N_ENGINES, timeout=1*24*60*60) # pause until `N_ENGINES` engines have connected or 1 day timeout 

print("Final IDs:", rc.ids)
print("Number of engines:", len(rc.ids)) # must match number of engines

Note that most of the code for computation with engines is in the notebook `Dynamical Critical Exponent from Normal Mode Topology.ipynb` and is imported here.

In [ ]:
dce_from_normal_modes_notebook = importlib.import_module("ipynb.fs.full.Dynamical Critical Exponent from Normal Mode Topology")

# Dynamical Critical Exponent from Species Projection Time Series

In [ ]:
%%px

import os
import math
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

from src.utils import *

In [ ]:
def compute_tau_single_run(L, observable_name, process_name, run_id, **observable_kwargs):
    rates_matrix = np.array([[0.0, 1.0, 0.1], [1.0, 0.0, 0.1], [2.1, 2.1, 0.0]])  # one KPZ mode, one diffusive mode

    observable_function = globals()[observable_name]

    mode_indices = [0, 1]

    n_samples = 6000
    sample_every = 25

    patch_divisor = 8
    patch_stride = 1
    tda_every = 2

    patch_window = max(8, int(L) // patch_divisor)

    # Optional reproducible seed.
    # seed = 10_000_000 + 1000 * int(L) + int(run_id)
    # np.random.seed(seed)

    process = MultiSpeciesExclusionProcess(dimension=3, density=[1/3, 1/3, 1/3], rates_matrix=rates_matrix, length=int(L))

    H = process.normal_mode_height_time_series(n_samples=n_samples, sample_every=sample_every)

    n_times = H.shape[0]
    tda_indices = np.arange(0, n_times, tda_every)

    taus_for_modes = []

    for mode_index in mode_indices:
        series = np.zeros(len(tda_indices), dtype=float)

        for j, t in enumerate(tda_indices):
            h_profile = H[t, :, mode_index]

            points = patch_point_cloud(h_profile, window=patch_window, stride=patch_stride)

            series[j] = observable_function(points, **observable_kwargs)

        C = autocorrelation(series)
        times = np.arange(len(C)) * sample_every * tda_every

        tau = relaxation_time(C, times)

        taus_for_modes.append(tau)

    return taus_for_modes

In [ ]:
dce_from_normal_modes_notebook.compute_tau_single_run = compute_tau_single_run

compute_tau_single_run = dce_from_normal_modes_notebook.compute_tau_single_run
compute_tau_for_L_batch = dce_from_normal_modes_notebook.compute_tau_for_L_batch

dce_from_normal_modes_notebook.push_functions_to_engines("compute_tau_single_run", "compute_tau_for_L_batch", "compute_tau_for_L")

## Strongly Coupled Topological Observables

In [ ]:
observable_name = "h0_total_persistence_from_points"
process_name = "raw species h0 total persistence"

L_values, taus = dce_from_normal_modes_notebook.engine_loop(process_name=process_name, observable_name=observable_name)

dce_from_normal_modes_notebook.fit(L_values, taus, process_name)

In [ ]:
observable_name = "h0_beta_curve_area_from_points"
process_name = "raw species h0 beta curve area"

L_values, taus = dce_from_normal_modes_notebook.engine_loop(process_name=process_name, observable_name=observable_name)

dce_from_normal_modes_notebook.fit(L_values, taus, process_name)

In [ ]:
observable_name = "h0_crocker_l2_norm_from_points"
process_name = "raw species h0 crocker l2 norm"

L_values, taus = dce_from_normal_modes_notebook.engine_loop(process_name=process_name, observable_name=observable_name)

dce_from_normal_modes_notebook.fit(L_values, taus, process_name)

In [ ]:
observable_name = "h0_entropy_from_points"
process_name = "raw species h0 entropy"

L_values, taus = dce_from_normal_modes_notebook.engine_loop(process_name=process_name, observable_name=observable_name)

dce_from_normal_modes_notebook.fit(L_values, taus, process_name)

## Weakly Coupled Topological Observables

In [ ]:
observable_name = "h1_total_persistence_from_points"
process_name = "raw species h1 total persistence"

L_values, taus = dce_from_normal_modes_notebook.engine_loop(process_name=process_name, observable_name=observable_name)

plot(L_values, taus, process_name)

In [ ]:
observable_name = "h1_beta_curve_area_from_points"
process_name = "raw species h1 beta curve area"

L_values, taus = engine_loop(process_name=process_name, observable_name=observable_name)

plot(L_values, taus, process_name)

In [ ]:
observable_name = "h1_max_persistence_from_points"
process_name = "raw species h1 maximum persistence"

L_values, taus = engine_loop(process_name=process_name, observable_name=observable_name)

plot(L_values, taus, process_name)

## Nonlinear/Uncoupled Topological Observables

## Shutting Down Engines and Slurm Process

In [ ]:
rc = ipp.Client(profile="default")
print("Engines before shutdown:", rc.ids)

rc.shutdown(hub=True, block=False)
rc.close()

print("Sent shutdown signal and closed client sockets.")

In [ ]:
cmd = "ps -u $USER -f | grep -E 'ipcontroller|ipengine|ipcluster' | grep -v grep"
result = subprocess.run(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, universal_newlines=True)

print(result.stdout if result.stdout else "No IPyParallel processes found.")

In [ ]:
result = subprocess.run("scancel --name=ipengines_tda", shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, universal_newlines=True)

if (result.stdout != ""):
    print("STDOUT:", result.stdout)
    
if (result.stderr != ""):
    print("STDERR:", result.stderr)
    
print("Return code:", result.returncode)